In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import zipfile
import mygene
import umap

from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler

from scipy import stats

from bioinfokit import analys, visuz
import bioinfokit

In [9]:
import sys
print("Python version:", sys.version) #expected: 3.11.16
print("pandas:", pd.__version__)      #expected: 2.2.3
print("bioinfokit:", bioinfokit.__version__)   #expected: 2.1.3

Python version: 3.11.16 (main, Sep  2 2026, 23:39:52) [GCC 15.3.0]
pandas: 2.2.3
bioinfokit: 2.1.3


In [10]:
#Read counts data
counts_tsv=pd.read_csv("SRP_counts.tsv", index_col=None, sep='\t')
print("Size of expression matrix: ", counts_tsv.shape[1]-1)
print("Number of genes: ", counts_tsv.shape[0])

just_counts_tsv=counts_tsv.iloc[:, 1:]

Size of expression matrix:  335
Number of genes:  43363


In [11]:
#Read in metadata
metadata=pd.read_csv("metadata.tsv", index_col=None, sep='\t')
types=metadata[['refinebio_accession_code', 'refinebio_subject']]  #so that we know control and not control
types_int=[0 if "control" in x 
           else 1 if 'bipolar disorder' in x 
           else 2 if 'major depression' in x
           else 3 
           for x in types['refinebio_subject']]  #0 label for control
metadata.head()

,refinebio_accession_code,experiment_accession,refinebio_age,refinebio_cell_line,refinebio_compound,refinebio_developmental_stage,refinebio_disease,refinebio_disease_stage,refinebio_genetic_information,refinebio_organism,...,refinebio_race,refinebio_sex,refinebio_source_archive_url,refinebio_source_database,refinebio_specimen_part,refinebio_subject,refinebio_time,refinebio_title,refinebio_treatment,MetaSRA_age at death
0,SRR3438555,SRP073813,NaN,NaN,NaN,NaN,NaN,NaN,NaN,HOMO_SAPIENS,...,caucasian,male,NaN,SRA,brain,ancg_control,NaN,X1834_AnCg_C_SL31501,NaN,40.0
1,SRR3438556,SRP073813,NaN,NaN,NaN,NaN,NaN,NaN,NaN,HOMO_SAPIENS,...,caucasian,male,NaN,SRA,brain,ancg_major depression,NaN,X2315_AnCg_M_SL31502,NaN,58.0
2,SRR3438557,SRP073813,NaN,NaN,NaN,NaN,NaN,NaN,NaN,HOMO_SAPIENS,...,caucasian,female,NaN,SRA,brain,ancg_bipolar disorder,NaN,X2566_AnCg_B_SL31503,NaN,56.0
3,SRR3438558,SRP073813,NaN,NaN,NaN,NaN,NaN,NaN,NaN,HOMO_SAPIENS,...,caucasian,male,NaN,SRA,brain,ancg_major depression,NaN,X3031_AnCg_M_SL31504,NaN,49.0
4,SRR3438559,SRP073813,NaN,NaN,NaN,NaN,NaN,NaN,NaN,HOMO_SAPIENS,...,caucasian,female,NaN,SRA,brain,nacc_schizophrenia,NaN,X2353_nAcc_S_SL31505,NaN,31.0


In [ ]:
len(set(metadata['refinbio_specimen_part']))

In [12]:
set(metadata['refinebio_subject'].tolist())

{'ancg_bipolar disorder',
 'ancg_control',
 'ancg_major depression',
 'ancg_schizophrenia',
 'dlpfc_bipolar disorder',
 'dlpfc_control',
 'dlpfc_major depression',
 'dlpfc_schizophrenia',
 'nacc_bipolar disorder',
 'nacc_control',
 'nacc_major depression',
 'nacc_schizophrenia'}

In [13]:
counts_tsv.head()

,Gene,SRR3438555,SRR3438556,SRR3438557,SRR3438558,SRR3438559,SRR3438560,SRR3438561,SRR3438562,SRR3438563,...,SRR3438896,SRR3438897,SRR3438898,SRR3438899,SRR3438900,SRR3438902,SRR3438903,SRR3438904,SRR3438905,SRR3438906
0,ENSG00000000003,1.942959,2.279117,2.088638,2.098454,2.399769,2.177106,2.681150,2.197950,1.910371,...,2.271737,2.438615,2.049946,2.056062,2.290635,1.988314,3.007927,2.023685,2.961496,2.251752
1,ENSG00000000005,-0.048304,0.265589,0.154746,0.337301,0.270970,-0.051191,0.086147,0.210492,-0.026680,...,0.252435,0.203276,0.203929,-0.074699,0.264386,0.205749,0.135252,-0.022101,-0.021257,0.300458
2,ENSG00000000419,2.104093,2.372098,2.472274,2.421515,2.226805,2.198438,2.105902,2.116410,2.081826,...,2.316789,2.086890,2.064837,2.169982,2.038607,2.023264,1.764969,1.670838,2.056917,2.742659
3,ENSG00000000457,2.324989,2.639704,2.372605,2.458040,2.204324,2.362276,2.245252,2.264167,2.363287,...,2.730702,2.378447,2.398443,2.193449,2.137209,2.088638,2.146767,2.251752,2.111357,3.085218
4,ENSG00000000460,1.378368,1.287078,1.374611,1.319985,1.501263,1.721814,1.474583,1.500013,1.526736,...,1.182988,1.463502,1.233033,1.242159,1.373331,1.491449,1.277179,1.260540,1.364809,1.524473


In [ ]:
mg=mygene.MyGeneInfo()
ensbml_ids=counts_tsv['Gene']
results=mg.querymany(ensbml_ids, scopes="ensembl.gene", fields="symbol", species="human")

In [ ]:
counts_tsv['Gene']=pd.DataFrame(results)[['symbol']]
counts_tsv.head()

### 2 Generating dim red plots

In [ ]:
### 2
labels=['Control' if x==0
        else 'Bipolar disorder' if x==1
        else 'Major depression' if x==2
        else 'Schizophrenia' for x in types_int]
counts=counts_tsv.iloc[:,1:].to_numpy().T

In [ ]:
### 2-PCA plots
pca=PCA(n_components=2)
pca_counts=pca.fit_transform(counts)

print("Explained variance percentage: ", pca.explained_variance_ratio_)

plt.figure(figsize=(6,6))
sns.scatterplot(x=pca_counts[:,0], y=pca_counts[:,1],
            hue=labels, palette='Dark2')
plt.xlabel("PC 1 (%.2f%%)" % (pca.explained_variance_ratio_[0]*100))
plt.ylabel("PC 2 (%.2f%%)" % (pca.explained_variance_ratio_[1]*100))
plt.title("PCA plot by group")
plt.legend(title='Status')
plt.show()

In [ ]:
### 2-TSNE plots
tsne=TSNE(n_components=2)
tsne_counts=tsne.fit_transform(counts)

plt.figure(figsize=(6,6))
sns.scatterplot(x=tsne_counts[:,0], y=tsne_counts[:,1],
            hue=labels, palette='Dark2')
plt.xlabel("TSNE 1")
plt.ylabel("TSNE 2")
plt.title("TSNE plot by group")
plt.legend(title='Status')
plt.show()

In [ ]:
### 2-UMAP plots
mapper=umap.UMAP().fit_transform(counts)

plt.figure(figsize=(6,6))
sns.scatterplot(x=mapper[:,0], y=mapper[:,1],
            hue=labels, palette='Dark2')
plt.xlabel("UMAP 1")
plt.ylabel("UMAP 2")
plt.title("UMAP plot by group")
plt.legend(title='Status')
plt.show()

### 2 Summary findings
1. While there are visible clusters formed in every plot, there are no clusters formed in between groups, meaning that many of the different types of brain disorders are closely clustered together.
2. UMAP shows the "clearest" distinction where as T-SNE and PCA clustes are more closely relaed to each other.
3. To add more

## 3. Differential analysis

### Calculating the log fold change

In [ ]:
max(counts_tsv.iloc[:,1].tolist())

In [ ]:
# divide between control and non-control (depression, bipolar, schiz)
# dataset has already been log transformed
control_inds=[ind for ind,i in enumerate(types_int) if i==0]
non_control_inds=[ind for ind,i in enumerate(types_int) if i!=0]

control_mean=just_counts_tsv.iloc[:, control_inds].mean(axis=1)
exp_mean=just_counts_tsv.iloc[:, non_control_inds].mean(axis=1)

log2FC=exp_mean-control_mean

ttest=stats.ttest_ind(just_counts_tsv.iloc[:, control_inds], 
                      just_counts_tsv.iloc[:, non_control_inds],
                      axis=1,
                      equal_var=False,
                      nan_policy="omit")
p_values=ttest.pvalue


volcano_df=(
    pd.DataFrame(
        {
            "log2FC": log2FC.to_numpy(),
            "pvalues": ttest.pvalue,
        },
        index=just_counts_tsv.index
    )
    .replace([np.inf, -np.inf], np.nan)
    .dropna(subset=["log2FC", "pvalues"])
    .copy()
)

In [ ]:
volcano_df.head()

### Plotting volcano plot

In [ ]:
visuz.GeneExpression.volcano(df=volcano_df,
                             lfc='log2FC',
                             pv='pvalues',
                             lfc_thr=(1,1),
                            pv_thr=(0.05, 0.05),
                            show=True)
#Note: if you have a linux OS, there will be warnings 
#saying 'findfont: Font family 'Arial' not found.' 
#but you can ignore this; the plot is at the bottom 

### 3-Getting top 50 genes

In [ ]:
top_50_idx = volcano_df['pvalues'].nsmallest(50).index
top_50_diff_genes = counts_tsv['Gene'].iloc[top_50_idx].tolist()

In [ ]:
top_50_diff_genes[:5]